# 传统机器学习基线：XGBoost、SVM-TS（TF-IDF/Word2Vec特征 + XGBoost分类）

In [1]:
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.svm import SVC
from xgboost import XGBClassifier
from sklearn.metrics import classification_report, accuracy_score

# 1. 加载数据
train_df = pd.read_csv('/mnt/workspace/FRD-CLIP/tmp/train_ready.csv')
test_df = pd.read_csv('/mnt/workspace/FRD-CLIP/tmp/test_ready.csv')

# 2. 特征提取 (TF-IDF)
tfidf = TfidfVectorizer(max_features=5000)
X_train = tfidf.fit_transform(train_df['text_clean'].astype(str))
X_test = tfidf.transform(test_df['text_clean'].astype(str))
y_train = train_df['label']
y_test = test_df['label']

# 3. 训练与评估函数
def evaluate_ml_model(model, name):
    model.fit(X_train, y_train)
    preds = model.predict(X_test)
    print(f"\n--- {name} Results ---")
    print(f"Accuracy: {accuracy_score(y_test, preds):.4f}")
    print(classification_report(y_test, preds, target_names=['real', 'fake']))

# 运行对比
evaluate_ml_model(SVC(kernel='linear', probability=True), "SVM (TF-IDF)")
evaluate_ml_model(XGBClassifier(n_estimators=100, learning_rate=0.1), "XGBoost (TF-IDF)")


--- SVM (TF-IDF) Results ---
Accuracy: 0.8392
              precision    recall  f1-score   support

        real       0.78      0.97      0.87       629
        fake       0.96      0.68      0.80       540

    accuracy                           0.84      1169
   macro avg       0.87      0.83      0.83      1169
weighted avg       0.86      0.84      0.83      1169


--- XGBoost (TF-IDF) Results ---
Accuracy: 0.8195
              precision    recall  f1-score   support

        real       0.75      1.00      0.86       629
        fake       1.00      0.61      0.76       540

    accuracy                           0.82      1169
   macro avg       0.87      0.80      0.81      1169
weighted avg       0.86      0.82      0.81      1169



# 深度学习序列基线：GRU、TextCNN（文本序列建模）

In [10]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from sklearn.metrics import classification_report, accuracy_score

# --- 数据预处理逻辑 ---
def get_dataloader(train_df, test_df, vocab_size=10000, max_len=100):
    # 极简分词与词表构建
    all_text = " ".join(train_df['text_clean'].astype(str)).split()
    vocab = {w: i+2 for i, (w, _) in enumerate(pd.Series(all_text).value_counts()[:vocab_size].items())}
    vocab['<PAD>'], vocab['<UNK>'] = 0, 1

    def encode(t):
        tokens = [vocab.get(w, 1) for w in str(t).split()]
        return tokens[:max_len] + [0] * max(0, max_len - len(tokens))

    class TextDataset(Dataset):
        def __init__(self, df):
            self.x = torch.tensor([encode(t) for t in df['text_clean']])
            self.y = torch.tensor(df['label'].values)
        def __len__(self): return len(self.y)
        def __getitem__(self, i): return self.x[i], self.y[i]

    return DataLoader(TextDataset(train_df), batch_size=32, shuffle=True), \
           DataLoader(TextDataset(test_df), batch_size=32), len(vocab)

# --- 模型定义 ---

class TextCNN(nn.Module):
    def __init__(self, vocab_size):
        super().__init__()
        self.emb = nn.Embedding(vocab_size, 128)
        self.convs = nn.ModuleList([nn.Conv1d(128, 100, k) for k in [3, 4, 5]])
        self.dropout = nn.Dropout(0.5)
        self.fc = nn.Linear(300, 2)
    def forward(self, x):
        x = self.emb(x).transpose(1, 2)
        x = [F.relu(conv(x)) for conv in self.convs]
        x = [F.max_pool1d(i, i.size(2)).squeeze(2) for i in x]
        return self.fc(self.dropout(torch.cat(x, 1)))

class GRUModel(nn.Module):
    def __init__(self, vocab_size):
        super().__init__()
        self.emb = nn.Embedding(vocab_size, 128)
        self.gru = nn.GRU(128, 128, bidirectional=True, batch_first=True)
        self.fc = nn.Linear(256, 2)
    def forward(self, x):
        _, hn = self.gru(self.emb(x))
        out = torch.cat([hn[-2], hn[-1]], dim=1) # 拼接双向最后隐藏状态
        return self.fc(out)

# --- 统一训练/评估函数 ---
def run_dl_experiment(model, train_loader, test_loader, name):
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    model.to(device)
    optimizer = torch.optim.AdamW(model.parameters(), lr=1e-3)
    criterion = nn.CrossEntropyLoss()

    for epoch in range(10):
        model.train()
        for x, y in train_loader:
            optimizer.zero_grad()
            loss = criterion(model(x.to(device)), y.to(device))
            loss.backward()
            optimizer.step()

    model.eval()
    y_true, y_pred = [], []
    with torch.no_grad():
        for x, y in test_loader:
            out = model(x.to(device))
            y_pred.extend(out.argmax(1).cpu().numpy())
            y_true.extend(y.numpy())

    print(f"\n{'='*20} {name} Results {'='*20}")
    print(f"Test Acc: {accuracy_score(y_true, y_pred):.4f}")
    print("Classification Report:")
    print(classification_report(y_true, y_pred, target_names=['real', 'fake']))

# 执行
train_loader, test_loader, v_size = get_dataloader(train_df, test_df)
run_dl_experiment(TextCNN(v_size), train_loader, test_loader, "TextCNN")
run_dl_experiment(GRUModel(v_size), train_loader, test_loader, "GRU")


==================== TextCNN Results ====================
Test Acc: 0.6775
Classification Report:
              precision    recall  f1-score   support

        real       0.63      0.99      0.77       629
        fake       0.96      0.31      0.47       540

    accuracy                           0.68      1169
   macro avg       0.79      0.65      0.62      1169
weighted avg       0.78      0.68      0.63      1169


==================== GRU Results ====================
Test Acc: 0.6775
Classification Report:
              precision    recall  f1-score   support

        real       0.63      1.00      0.77       629
        fake       0.99      0.31      0.47       540

    accuracy                           0.68      1169
   macro avg       0.81      0.65      0.62      1169
weighted avg       0.79      0.68      0.63      1169

